In [1]:
import os
import numpy as np
import pickle
import torch
from argparse import ArgumentParser
from tqdm import tqdm
import glob

from articulate.model import ParametricModel
from articulate import math
from config import paths, datasets


In [ ]:
# specify target FPS
TARGET_FPS = 30

# left wrist, right wrist, left thigh, right thigh, head, pelvis
vi_mask = torch.tensor([1961, 5424, 876, 4362, 411, 3021])
ji_mask = torch.tensor([18, 19, 1, 2, 15, 0])
body_model = ParametricModel(paths.smpl_file)
def _syn_acc(v, smooth_n=4):
    """Synthesize accelerations from vertex positions."""
    mid = smooth_n // 2
    scale_factor = TARGET_FPS ** 2 

    acc = torch.stack([(v[i] + v[i + 2] - 2 * v[i + 1]) * scale_factor for i in range(0, v.shape[0] - 2)])
    acc = torch.cat((torch.zeros_like(acc[:1]), acc, torch.zeros_like(acc[:1])))

    if mid != 0:
        acc[smooth_n:-smooth_n] = torch.stack(
            [(v[i] + v[i + smooth_n * 2] - 2 * v[i + smooth_n]) * scale_factor / smooth_n ** 2
             for i in range(0, v.shape[0] - smooth_n * 2)])
    return acc


In [ ]:
def process_amass():
    def _foot_ground_probs(joint):
        """Compute foot-ground contact probabilities."""
        dist_lfeet = torch.norm(joint[1:, 10] - joint[:-1, 10], dim=1)
        dist_rfeet = torch.norm(joint[1:, 11] - joint[:-1, 11], dim=1)
        lfoot_contact = (dist_lfeet < 0.008).int()
        rfoot_contact = (dist_rfeet < 0.008).int()
        lfoot_contact = torch.cat((torch.zeros(1, dtype=torch.int), lfoot_contact))
        rfoot_contact = torch.cat((torch.zeros(1, dtype=torch.int), rfoot_contact))
        return torch.stack((lfoot_contact, rfoot_contact), dim=1)

    # enable skipping processed files
    try:
        processed = [fpath.name for fpath in (paths.processed_datasets).iterdir()]
    except FileNotFoundError:
        processed = []

    for ds_name in datasets.amass_datasets:
        # skip processed 
        if f"{ds_name}.pt" in processed:
            continue

        data_pose, data_trans, data_beta, length = [], [], [], []
        print("\rReading", ds_name)

        for npz_fname in tqdm(sorted(glob.glob(os.path.join(paths.raw_amass, ds_name, "*/*_poses.npz")))):
            try: cdata = np.load(npz_fname)
            except: continue

            framerate = int(cdata['mocap_framerate'])
            if framerate not in [120, 60, 59]:
                continue

            # enable downsampling
            step = max(1, round(framerate / TARGET_FPS))

            data_pose.extend(cdata['poses'][::step].astype(np.float32))
            data_trans.extend(cdata['trans'][::step].astype(np.float32))
            data_beta.append(cdata['betas'][:10])
            length.append(cdata['poses'][::step].shape[0])

        if len(data_pose) == 0:
            print(f"AMASS dataset, {ds_name} not supported")
            continue

        length = torch.tensor(length, dtype=torch.int)
        print(f"Total frames in {ds_name}: {length}")
        shape = torch.tensor(np.asarray(data_beta, np.float32))
        tran = torch.tensor(np.asarray(data_trans, np.float32))
        pose = torch.tensor(np.asarray(data_pose, np.float32)).view(-1, 52, 3)

        # include the left and right index fingers in the pose
        pose[:, 23] = pose[:, 37]     # right hand 
        pose = pose[:, :24].clone()   # only use body + right and left fingers

        # align AMASS global frame with DIP
        amass_rot = torch.tensor([[[1, 0, 0], [0, 0, 1], [0, -1, 0.]]])
        tran = amass_rot.matmul(tran.unsqueeze(-1)).view_as(tran)
        pose[:, 0] = math.rotation_matrix_to_axis_angle(
            amass_rot.matmul(math.axis_angle_to_rotation_matrix(pose[:, 0])))

        print("Synthesizing IMU accelerations and orientations")
        b = 0
        out_pose, out_shape, out_tran, out_joint, out_vrot, out_vacc, out_contact = [], [], [], [], [], [], []
        for i, l in tqdm(list(enumerate(length))):
            if l <= 12: b += l; print("\tdiscard one sequence with length", l); continue
            p = math.axis_angle_to_rotation_matrix(pose[b:b + l]).view(-1, 24, 3, 3)
            grot, joint, vert = body_model.forward_kinematics(p, shape[i], tran[b:b + l], calc_mesh=True)

            out_pose.append(p.clone())  # N, 24, 3, 3
            out_tran.append(tran[b:b + l].clone())  # N, 3
            out_shape.append(shape[i].clone())  # 10
            out_joint.append(joint[:, :24].contiguous().clone())  # N, 24, 3
            out_vacc.append(_syn_acc(vert[:, vi_mask]))  # N, 6, 3
            out_contact.append(_foot_ground_probs(joint).clone()) # N, 2

            out_vrot.append(grot[:, ji_mask])  # N, 6, 3, 3
            b += l


def create_directories():
    paths.processed_datasets.mkdir(exist_ok=True, parents=True)
    paths.eval_dir.mkdir(exist_ok=True, parents=True)


In [ ]:
create_directories()

In [ ]:
process_amass()

In [ ]:
for ds_name in datasets.amass_datasets:
    print(ds_name)

In [ ]:
for npz_fname in tqdm(sorted(glob.glob(os.path.join(paths.raw_amass, ds_name, "*/*_poses.npz")))):
    print(npz_fname)

In [ ]:
ds_name= 'ACCAD'

In [ ]:
base_path = os.path.join(paths.raw_amass, ds_name)
print("Searching in:", base_path)
print("Folder exists:", os.path.exists(base_path))
print("Contents:", os.listdir(base_path) if os.path.exists(base_path) else "No folder")

In [ ]:
for i in tqdm(sorted(glob.glob(os.path.join(paths.raw_amass, ds_name, "*/*_poses.npz")))):
    print(i)

In [ ]:
def debug_amass_accad_two_sequences():


    ds_name = 'ACCAD'
    sequence_count = 0
    pose_np, trans_np, data_beta, length = [], [], [], []
    print("\rReading", ds_name)

    for npz_fname in tqdm(sorted(glob.glob(os.path.join(paths.raw_amass, ds_name, "*/*_poses.npz")))):
        try: cdata = np.load(npz_fname)
        except: continue

        framerate = int(cdata['mocap_framerate'])
        if framerate not in [120, 60, 59]:
            continue

        step = max(1, round(framerate / TARGET_FPS))

        pose_np = cdata['poses'][::step].astype(np.float32)
        trans_np = cdata['trans'][::step].astype(np.float32)

        print("\n==============================")
        print(f"SEQUENCE {sequence_count + 1}")
        print("==============================")
        print("Raw pose shape:", pose_np.shape)     # (N, 156)
        print("Raw trans shape:", trans_np.shape)   # (N, 3)

        pose = torch.tensor(pose_np).view(-1, 52, 3)
        tran = torch.tensor(trans_np)

        print("Reshaped pose:", pose.shape)        # (N, 52, 3)

        # Keep first 24 joints
        pose[:, 23] = pose[:, 37]
        pose = pose[:, :24].clone()

        print("Trimmed pose:", pose.shape)         # (N, 24, 3)

        print("\nSample values:")
        print("First frame root axis-angle:", pose[0, 0])
        print("First 3 translations:\n", tran[:3])

        sequence_count += 1

        if sequence_count == 2:
            break

In [ ]:
debug_amass_accad_two_sequences()

In [ ]:

def process_dipimu(split="test"):
    """Preprocess DIP for finetuning and evaluation."""
    imu_mask = [7, 8, 9, 10, 0, 2]

    test_split = ['s_09', 's_10']
    train_split = ['s_01', 's_02', 's_03', 's_04', 's_05', 's_06', 's_07', 's_08']
    subjects = train_split if split == "train" else test_split
     
    # left wrist, right wrist, left thigh, right thigh, head, pelvis
    vi_mask = torch.tensor([1961, 5424, 876, 4362, 411, 3021])
    ji_mask = torch.tensor([18, 19, 1, 2, 15, 0])

    # enable downsampling
    step = max(1, round(60 / TARGET_FPS))

    accs, oris, poses, trans, shapes, joints = [], [], [], [], [], []
    for subject_name in subjects:
        for motion_name in os.listdir(os.path.join(paths.raw_dip, subject_name)):
            try:
                path = os.path.join(paths.raw_dip, subject_name, motion_name)
                print(f"Processing: {subject_name}/{motion_name}")
                data = pickle.load(open(path, 'rb'), encoding='latin1')
                acc = torch.from_numpy(data['imu_acc'][:, imu_mask]).float()
                ori = torch.from_numpy(data['imu_ori'][:, imu_mask]).float()
                pose = torch.from_numpy(data['gt']).float()
                print("Original shapes - acc:", acc.shape, "ori:", ori.shape, "pose:", pose.shape)
                # fill nan with nearest neighbors
                for _ in range(4):
                    acc[1:].masked_scatter_(torch.isnan(acc[1:]), acc[:-1][torch.isnan(acc[1:])])
                    ori[1:].masked_scatter_(torch.isnan(ori[1:]), ori[:-1][torch.isnan(ori[1:])])
                    acc[:-1].masked_scatter_(torch.isnan(acc[:-1]), acc[1:][torch.isnan(acc[:-1])])
                    ori[:-1].masked_scatter_(torch.isnan(ori[:-1]), ori[1:][torch.isnan(ori[:-1])])

                # enable downsampling
                acc = acc[6:-6:step].contiguous()
                ori = ori[6:-6:step].contiguous()
                pose = pose[6:-6:step].contiguous()
                print("pose first",pose.shape)
                shape = torch.ones((10))
                tran = torch.zeros(pose.shape[0], 3) # dip-imu does not contain translations
                print("tran shape",tran.shape)
                if torch.isnan(acc).sum() == 0 and torch.isnan(ori).sum() == 0 and torch.isnan(pose).sum() == 0:
                    accs.append(acc.clone())
                    oris.append(ori.clone())
                    trans.append(tran.clone())  
                    shapes.append(shape.clone()) # default shape
                    
                    # forward kinematics to get the joint position
                    p = math.axis_angle_to_rotation_matrix(pose).reshape(-1, 24, 3, 3)
                    print("pose after reshape",p.shape)
                    grot, joint, vert = body_model.forward_kinematics(p, shape, tran, calc_mesh=True)
                    poses.append(p.clone())
                    joints.append(joint)
                else:
                    print(f"DIP-IMU: {subject_name}/{motion_name} has too much nan! Discard!")
                # print("dsgasdg",trans.shape, shape.shape, poses.shape, joints.shape)
                print(len(poses))
                print("=====================================")
                print("pose shape",poses[1].shape)
            except Exception as e:
                print(f"Error processing the file: {path}.", e)


In [ ]:
process_dipimu()

In [ ]:
step = max(1, round(60 / TARGET_FPS))
print("step", step  )

In [40]:
from data import PoseDataset
import torch
from torch.utils.data import Dataset, DataLoader, random_split
dataset_name = "dip"
# dataset = PoseDataset( evaluate=dataset_name)   # loads processed_datasets

# dataset = PoseDataset(fold='test', evaluate=dataset_name)   # loads processed_datasets
dataset = PoseDataset(fold='train', finetune=None)
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])



Loading datasets for TRAIN mode (finetune=None, evaluate=None)
Datasets: ['ACCAD.pt', 'BioMotionLab_NTroje.pt', 'BMLhandball.pt', 'BMLmovi.pt', 'CMU.pt', 'DanceDB.pt', 'DFaust_67.pt', 'dip_test.pt', 'dip_train.pt', 'Eyes_Japan_Dataset.pt', 'HUMAN4D.pt', 'HumanEva.pt', 'MPI_HDM05.pt', 'MPI_Limits.pt', 'MPI_mosh.pt', 'SFU.pt', 'SSM_synced.pt', 'TCD_handMocap.pt', 'TotalCapture.pt', 'Transitions_mocap.pt']



  0%|          | 0/20 [00:00<?, ?it/s]c:\deepika\2_mobileposer\MobilePoser\mobileposer\data.py:55: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  file_data = torch.load(data_

In [3]:
import torch.nn as nn
from config import *
from helpers import * 


In [41]:
def pad_seq(batch):
    """Pad sequences to same length for RNN."""
    def _pad(sequence):
        padded = nn.utils.rnn.pad_sequence(sequence, batch_first=True)
        lengths = [seq.shape[0] for seq in sequence]
        return padded, lengths

    inputs, poses, joints, trans = zip(*[(item[0], item[1], item[2], item[3]) for item in batch])
    inputs, input_lengths = _pad(inputs)
    poses, pose_lengths = _pad(poses)
    joints, joint_lengths = _pad(joints)
    trans, tran_lengths = _pad(trans)
    
    outputs = {'poses': poses, 'joints': joints, 'trans': trans}
    output_lengths = {'poses': pose_lengths, 'joints': joint_lengths, 'trans': tran_lengths}

    if len(batch[0]) > 5: # include velocity and foot contact, if available
        vels, foots = zip(*[(item[4], item[5]) for item in batch])

        # foot contact 
        foot_contacts, foot_contact_lengths = _pad(foots)
        outputs['foot_contacts'], output_lengths['foot_contacts'] = foot_contacts, foot_contact_lengths

        # root velocities
        vels, vel_lengths = _pad(vels)
        outputs['vels'], output_lengths['vels'] = vels, vel_lengths

    return (inputs, input_lengths), (outputs, output_lengths)
def _dataloader(dataset):
        return DataLoader(
            dataset, 
            batch_size=256, 
            collate_fn=pad_seq, 
            num_workers=0, 
            shuffle=True, 
            drop_last=True
        )

In [42]:
train_dataloader_ = _dataloader(train_dataset)

In [43]:
len(train_dataloader_)

1568

In [44]:
for (inputs, input_lengths), (outputs, output_lengths) in train_dataloader_:
    print("INPUT SHAPE:", inputs.shape)
    print("POSES SHAPE:", outputs['poses'].shape)
    print("JOINTS SHAPE:", outputs['joints'].shape)
    print("TRANS SHAPE:", outputs['trans'].shape)
    print("INPUT LENGTHS:", input_lengths[:5])  # first 5 lengths
    break  # only look at first batch

INPUT SHAPE: torch.Size([256, 125, 60])
POSES SHAPE: torch.Size([256, 125, 144])
JOINTS SHAPE: torch.Size([256, 125, 24, 3])
TRANS SHAPE: torch.Size([256, 125, 3])
INPUT LENGTHS: [125, 125, 125, 125, 125]


In [38]:
for (inputs, input_lengths), (outputs, output_lengths) in train_dataloader_:
    break

In [39]:
for i in range(3):
    print(f"\n===== Sample {i} =====")
    
    print("Input shape: ", inputs[i].shape)
    print("Pose shape:  ", outputs['poses'][i].shape)
    print("Joint shape: ", outputs['joints'][i].shape)
    print("Trans shape: ", outputs['trans'][i].shape)

    print("Real length:", input_lengths[i])


===== Sample 0 =====
Input shape:  torch.Size([2708, 60])
Pose shape:   torch.Size([2708, 144])
Joint shape:  torch.Size([2708, 24, 3])
Trans shape:  torch.Size([2708, 3])
Real length: 750

===== Sample 1 =====
Input shape:  torch.Size([2708, 60])
Pose shape:   torch.Size([2708, 144])
Joint shape:  torch.Size([2708, 24, 3])
Trans shape:  torch.Size([2708, 3])
Real length: 1020

===== Sample 2 =====
Input shape:  torch.Size([2708, 60])
Pose shape:   torch.Size([2708, 144])
Joint shape:  torch.Size([2708, 24, 3])
Trans shape:  torch.Size([2708, 3])
Real length: 1062


In [17]:
val_dataloader_ = _dataloader(val_dataset)

In [18]:
len(val_dataloader_)

23

In [19]:
for i, sample in enumerate(val_dataloader_.dataset):
    print(sample[0].shape)  # input shape
    print(sample[1].shape)  # pose shape
    print(sample[3].shape)
    # print("pose",sample[1]['poses'])    
    print(i)
    print("=====================================")

torch.Size([1448, 60])
torch.Size([1448, 144])
torch.Size([1448, 3])
0
torch.Size([2373, 60])
torch.Size([2373, 144])
torch.Size([2373, 3])
1
torch.Size([1804, 60])
torch.Size([1804, 144])
torch.Size([1804, 3])
2
torch.Size([1020, 60])
torch.Size([1020, 144])
torch.Size([1020, 3])
3
torch.Size([1136, 60])
torch.Size([1136, 144])
torch.Size([1136, 3])
4
torch.Size([1870, 60])
torch.Size([1870, 144])
torch.Size([1870, 3])
5
torch.Size([1804, 60])
torch.Size([1804, 144])
torch.Size([1804, 3])
6
torch.Size([2708, 60])
torch.Size([2708, 144])
torch.Size([2708, 3])
7
torch.Size([2373, 60])
torch.Size([2373, 144])
torch.Size([2373, 3])
8
torch.Size([588, 60])
torch.Size([588, 144])
torch.Size([588, 3])
9
torch.Size([588, 60])
torch.Size([588, 144])
torch.Size([588, 3])
10
torch.Size([2799, 60])
torch.Size([2799, 144])
torch.Size([2799, 3])
11
torch.Size([1136, 60])
torch.Size([1136, 144])
torch.Size([1136, 3])
12
torch.Size([2708, 60])
torch.Size([2708, 144])
torch.Size([2708, 3])
13
torch.Si

In [ ]:
from data import PoseDataset
import torch

ds = PoseDataset(fold='train', finetune=None)   # loads processed_datasets

dataset = PoseDataset(fold='train', finetune=None)
train_size = int(0.9 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])



pairs = []   # list of (pose_seq, tran_seq) per window/sequence
for i in range(len(ds)):
    item = ds[i]
    # training dataset returns (imu, pose, joint, tran, vel, contact)
    imu, pose, joint, tran = item[:4]
    # pose: (T, 6 * num_pred_joints) , tran: (T, 3)
    pairs.append((imu,pose, tran))


# # example: concatenate all sequences (if lengths match) or keep as list
all_imus = torch.cat([i for i,p, t in pairs], dim=0)
all_poses = torch.cat([p for i,p, t in pairs], dim=0)
all_trans  = torch.cat([t for i,p, t in pairs], dim=0)

device = "cpu"
x0 = torch.cat([all_poses.to(device), all_trans.to(device)], dim=-1)  # (B, 147)